# ML-09 — Validation and Research Claim Audit

> **Author:** Ahmed El-Dasoky · FlyRank ML Internship · Applied Search Intelligence track
> **Lane:** Refresh / Content Opportunity Scoring
> **Data:** bundled `data/raw/content_refresh_anonymized.csv` (30,000 pseudonymized pages, 44 columns) + the reference pipeline in `scripts/` and its committed outputs.

Run top to bottom — every code cell reads only public, bundled data and prints numbers, never raw queries or client names.


# ML-09 — Validation and Research Claim Audit


## 1. Two paper findings + my methodology questions
- Finding A: 'visibility decays fastest for mid-position pages' → my question: my client-holdout split mirrors production? (clients vary in history → grouped split is mandatory).
- Finding B: 'refresh recovers CTR for declining pages' → my question: my model ranks *decline likelihood*, it does NOT estimate recovery — so any claim must stay 'recommend for review', never 'will recover'.
- Methodology question I had to answer: does random-row CV inflate my metrics? → Yes — that is exactly why I use **client-grouped** splits.

## 2. My model under an honest split (before/after)

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_down"] = df["trend_direction"].eq("down").astype(int)
feats = ["impressions_90d","clicks_90d","sessions_90d","users_90d","engaged_sessions_90d","scroll_events_90d",
         "days_with_impressions","search_volume","word_count","content_age_days","days_since_last_update","ctr",
         "avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
X = df[feats].fillna(df[feats].median()); y = df["is_down"]

def auc_on(split):
    m = HistGradientBoostingClassifier(random_state=42, max_iter=60)
    tr, te = split
    m.fit(X.iloc[tr], y.iloc[tr])
    return roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te])[:,1])

# random-row split (optimistic reference)
from sklearn.model_selection import train_test_split
tr0, te0 = train_test_split(np.arange(len(df)), test_size=.2, random_state=1)
random_auc = auc_on((tr0, te0))

# client-grouped split (honest)
gss = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42)
tr1, te1 = next(gss.split(X, y, groups=df["client_id"]))
grouped_auc = auc_on((tr1, te1))

print(f"AUC random-row split   : {random_auc:.3f} (optimistic)")
print(f"AUC client-grouped split: {grouped_auc:.3f} (honest)")

## 3. Leakage audit
Final feature set re-checked: only backward-looking aggregates + static metadata; label (trend_direction) excluded from features; no provider/model columns; content_id/client_id used only for grouping. (Same checks as ML-05 — re-run here; expected 0 leaks.)

In [ ]:
# leakage audit on the ACTUAL feature set used by the model
feat_set = ["impressions_90d","clicks_90d","sessions_90d","users_90d","engaged_sessions_90d","scroll_events_90d",
            "days_with_impressions","search_volume","word_count","content_age_days","days_since_last_update","ctr",
            "avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
banned = [f for f in feat_set if f in ("provider_used","model_used","trend_direction")]
future = [f for f in feat_set if any(k in f for k in ("_next","_future","_lead"))]
print("banned in feature set:", banned, "| future-window features:", future)
print("leak audit clean:", not banned and not future)

## 4. Claim rewrite

**Bold before:** 'My model identifies the pages that will recover after a refresh.'
**Safe after:** 'Within the bundled 90-day snapshot, a client-grouped model ranks existing content for *review* — pages scored high were declining more often than not in the observed window (decision-support; no claim of recovery).'

## Self-check

- [x] Two paper findings discussed
- [x] Before/after honest-split numbers
- [x] Leakage audit re-run
- [x] Claim rewritten in safe language